# Module 26 – Sales Forecasting Case Study

## Objective
Analyze sales data over time, prepare it for forecasting, build a predictive model, evaluate it using **RMSE, MAE and R²**, and summarize the findings.

### Model used
This notebook uses **Linear Regression with time-based features**. It is beginner-friendly and does not require installing additional forecasting packages such as Prophet.

### Dataset
Use a Kaggle sales dataset containing at least:
- a date column
- a sales/revenue column

> **Important:** The code below tries to automatically identify common date and sales column names. If your dataset uses different names, change the two variables in the loading section.


## 1. Import Libraries

We use:
- `pandas` and `numpy` for data handling
- `matplotlib` and `seaborn` for visualization
- `scikit-learn` for Linear Regression and evaluation metrics


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")


## 2. Load the Dataset

### In Google Colab
Run the cell and upload your CSV file.

If you are using Jupyter Notebook on your computer, you can replace the upload code with:

```python
df = pd.read_csv("your_file_name.csv")
```


In [ ]:
# Google Colab upload
try:
    from google.colab import files
    uploaded = files.upload()
    file_name = next(iter(uploaded))
except ImportError:
    file_name = "sales.csv"   # Change this for local Jupyter

df = pd.read_csv(file_name)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
df.head()


## 3. Identify Date and Sales Columns

The code checks common column names. If it cannot find your columns, manually set:

```python
date_col = "YourDateColumn"
sales_col = "YourSalesColumn"
```


In [ ]:
# Display available columns
print("Columns in dataset:")
print(df.columns.tolist())

# Try to detect common date and sales columns
date_candidates = [
    "Date", "date", "Order Date", "Order_Date", "Sales Date",
    "sales_date", "Invoice Date", "Invoice_Date", "Transaction Date"
]

sales_candidates = [
    "Sales", "sales", "Revenue", "revenue", "Amount",
    "amount", "Total Sales", "Total_Sales", "Sales Amount"
]

date_col = next((c for c in date_candidates if c in df.columns), None)
sales_col = next((c for c in sales_candidates if c in df.columns), None)

print("Detected date column:", date_col)
print("Detected sales column:", sales_col)

# If needed, uncomment and edit these:
# date_col = "Order Date"
# sales_col = "Sales"

if date_col is None or sales_col is None:
    raise ValueError(
        "Could not automatically detect the date/sales column. "
        "Please set date_col and sales_col manually using the column names printed above."
    )


## 4. Basic Dataset Exploration

In [ ]:
print("First 5 rows:")
display(df.head())

print("\nDataset information:")
df.info()

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nSummary statistics:")
display(df.describe(include="all"))


## 5. Data Preprocessing

Steps:
1. Convert the date column to datetime.
2. Convert sales to numeric.
3. Remove rows with invalid dates.
4. Fill missing sales values using the median.
5. Sort the dataset chronologically.


In [ ]:
# Convert date column
df[date_col] = pd.to_datetime(df[date_col], errors="coerce")

# Convert sales column to numeric
df[sales_col] = pd.to_numeric(df[sales_col], errors="coerce")

# Remove rows where date is invalid
df = df.dropna(subset=[date_col]).copy()

# Fill missing sales with median
df[sales_col] = df[sales_col].fillna(df[sales_col].median())

# Remove duplicate rows
df = df.drop_duplicates().copy()

# Sort chronologically
df = df.sort_values(date_col).reset_index(drop=True)

print("After preprocessing:", df.shape)
print("Missing date values:", df[date_col].isnull().sum())
print("Missing sales values:", df[sales_col].isnull().sum())

display(df[[date_col, sales_col]].head())


## 6. Extract Time-Based Features

Time series models can learn patterns from calendar information. We extract:
- Year
- Month
- Day
- Day of week
- Week of year
- Quarter
- A sequential time index

The **time index** captures the overall trend from earlier observations to later observations.


In [ ]:
df["Year"] = df[date_col].dt.year
df["Month"] = df[date_col].dt.month
df["Day"] = df[date_col].dt.day
df["DayOfWeek"] = df[date_col].dt.dayofweek
df["WeekOfYear"] = df[date_col].dt.isocalendar().week.astype(int)
df["Quarter"] = df[date_col].dt.quarter

# Sequential time index
df["TimeIndex"] = np.arange(len(df))

display(df[[date_col, sales_col, "Year", "Month", "Day", "DayOfWeek", "Quarter", "TimeIndex"]].head())


## 7. Check Sales Trend Over Time

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(df[date_col], df[sales_col])
plt.title("Sales Trend Over Time")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### Monthly Sales Trend

If the dataset contains enough observations, monthly aggregation can make seasonal patterns easier to see.


In [ ]:
monthly_sales = (
    df.set_index(date_col)[sales_col]
    .resample("ME")
    .sum()
)

plt.figure(figsize=(14, 6))
plt.plot(monthly_sales.index, monthly_sales.values)
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Total Sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 8. Check Possible Seasonality

In [ ]:
monthly_average = df.groupby("Month")[sales_col].mean()

plt.figure(figsize=(10, 5))
plt.bar(monthly_average.index, monthly_average.values)
plt.title("Average Sales by Month")
plt.xlabel("Month")
plt.ylabel("Average Sales")
plt.xticks(range(1, 13))
plt.tight_layout()
plt.show()

print("Average sales by month:")
display(monthly_average.to_frame("Average_Sales"))


## 9. Prepare Features and Target

We use:
- `TimeIndex` → overall trend
- `Year`
- `Month`
- `Day`
- `DayOfWeek`
- `WeekOfYear`
- `Quarter`

The target variable is the sales value.


In [ ]:
feature_cols = [
    "TimeIndex", "Year", "Month", "Day",
    "DayOfWeek", "WeekOfYear", "Quarter"
]

X = df[feature_cols]
y = df[sales_col]

print("Features:")
display(X.head())

print("Target:")
display(y.head())


## 10. Chronological Train-Test Split

For forecasting, we **must not randomly shuffle the data**.

The earlier observations are used for training and the later observations are used for testing. This better represents a real forecasting situation.


In [ ]:
# Use the first 80% for training and the final 20% for testing
split_index = int(len(df) * 0.80)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining period:")
print(df[date_col].iloc[0], "to", df[date_col].iloc[split_index - 1])

print("\nTesting period:")
print(df[date_col].iloc[split_index], "to", df[date_col].iloc[-1])


## 11. Train Linear Regression Model

In [ ]:
model = LinearRegression()

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Model training completed.")


## 12. Evaluate the Forecast

We calculate:
- **MAE (Mean Absolute Error):** average absolute prediction error.
- **RMSE (Root Mean Squared Error):** penalizes larger errors more strongly.
- **R² Score:** shows how much variation in sales is explained by the model.

Lower MAE/RMSE is generally better. Higher R² is generally better.


In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"R²   : {r2:.4f}")


## 13. Actual vs Predicted Sales

In [ ]:
comparison = pd.DataFrame({
    "Date": df[date_col].iloc[split_index:].values,
    "Actual_Sales": y_test.values,
    "Predicted_Sales": y_pred
})

display(comparison.head(10))

plt.figure(figsize=(14, 6))
plt.plot(comparison["Date"], comparison["Actual_Sales"], label="Actual Sales")
plt.plot(comparison["Date"], comparison["Predicted_Sales"], label="Predicted Sales")
plt.title("Actual vs Predicted Sales")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 14. Forecast Error Analysis

Looking at the residuals helps us understand where the model is making larger mistakes.


In [ ]:
comparison["Error"] = comparison["Actual_Sales"] - comparison["Predicted_Sales"]

plt.figure(figsize=(14, 5))
plt.plot(comparison["Date"], comparison["Error"])
plt.axhline(0, linestyle="--")
plt.title("Forecast Errors Over Time")
plt.xlabel("Date")
plt.ylabel("Actual - Predicted")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("Average error:", comparison["Error"].mean())
print("Maximum absolute error:", comparison["Error"].abs().max())


## 15. Model Coefficients

The coefficients show how the Linear Regression model uses each feature.

**Note:** A coefficient does not automatically mean that the feature causes sales to increase or decrease. It represents the model's learned relationship while the other features are considered.


In [ ]:
coefficients = pd.DataFrame({
    "Feature": feature_cols,
    "Coefficient": model.coef_
}).sort_values("Coefficient", ascending=False)

display(coefficients)


## 16. Brief Analysis and Findings

### Dataset observations
- The dataset was converted into a chronological time series.
- Missing values were handled before modeling.
- Date information was transformed into useful calendar features.
- The sales trend and monthly patterns were visualized.

### Modeling observations
- A chronological 80:20 train-test split was used to avoid future information leaking into training.
- Linear Regression was trained using time and calendar features.
- The model was evaluated using MAE, RMSE and R².

### Interpretation
After running the notebook, compare the actual and predicted plots and the evaluation metrics.

**Write your final observations here:**
1. Sales generally show __________________________ over time.
2. The highest average sales occurred in month __________________.
3. The model achieved an MAE of __________________.
4. The model achieved an RMSE of __________________.
5. The model achieved an R² score of __________________.
6. The actual-vs-predicted plot shows that the model __________________________.
7. Possible improvements include using lag features, rolling averages, Random Forest/Gradient Boosting, ARIMA, Prophet, or another time-series model.


## 17. Conclusion

This case study demonstrated a complete beginner-friendly sales forecasting workflow:

**Load Data → Explore → Clean → Convert Dates → Feature Engineering → Chronological Split → Train Model → Forecast → Evaluate → Analyze**

Linear Regression provides a simple baseline for forecasting. More advanced forecasting methods can be tested later when the basic workflow is understood.


# Submission Checklist

Before submitting:

- [ ] Dataset imported successfully
- [ ] Date column converted to datetime
- [ ] Missing values checked and handled
- [ ] Time-based features created
- [ ] Sales trend visualized
- [ ] Seasonality/monthly pattern checked
- [ ] Chronological train-test split used
- [ ] Linear Regression model trained
- [ ] MAE calculated
- [ ] RMSE calculated
- [ ] R² calculated
- [ ] Actual vs predicted graph included
- [ ] Brief analysis completed with your actual results
- [ ] Notebook saved as `.ipynb`
- [ ] Notebook uploaded to Google Colab
- [ ] Colab sharing changed to **Anyone with the link → Viewer**
- [ ] Colab link submitted
